# Week 3.2: Evaluations - Testing Your AI

## Key Insight: How to write 'Unit Tests' for an AI

You need to TEST your digital twin like you test code!

## What You'll Build
- Test datasets for your AI
- Evaluation metrics (accuracy, hallucination)
- Automated testing framework
- Quality assurance for your twin

In [ ]:
from typing import List, Dict, Any
import json

print('Week 3.2: Evaluations & Testing')

print('Goal: Ensure your digital twin works correctly!')

Week 3.2: Evaluations & Testing
Goal: Ensure your digital twin works correctly!


## Part 1: Creating Test Cases

Test cases have: input, expected output, pass criteria.

In [ ]:
class TestCase:
    '''Single test case for AI evaluation.'''

    def __init__(self, input_text: str, expected_output: str,
                 criteria: str = 'exact_match'):
        self.input = input_text
        self.expected = expected_output
        self.criteria = criteria

    def evaluate(self, actual_output: str) -> Dict[str, Any]:
        '''Evaluate if test passed.'''
        if self.criteria == 'exact_match':
            passed = actual_output.strip() == self.expected.strip()
        elif self.criteria == 'contains':
            passed = self.expected.lower() in actual_output.lower()
        elif self.criteria == 'keyword':
            # Check if key words are present
            keywords = self.expected.split()
            passed = all(kw.lower() in actual_output.lower() for kw in keywords)
        else:
            passed = False

        return {
            'passed': passed,
            'input': self.input,
            'expected': self.expected,
            'actual': actual_output,
            'criteria': self.criteria
        }

# Example test cases
test_cases = [
    TestCase(
        "What is 2 + 2?",
        "4",
        "contains"
    ),
    TestCase(
        "Who created you?",
        "I am a digital twin",
        "contains"
    ),
    TestCase(
        "What's the weather?",
        "I don't have access to current weather",
        "contains"
    )
]

print(f'Created {len(test_cases)} test cases')

Created 3 test cases


## Part 2: Evaluation Metrics

Measure quality of AI responses.

In [ ]:
class EvaluationMetrics:
    '''Calculate metrics for AI outputs.'''

    @staticmethod
    def accuracy(results: List[Dict]) -> float:
        '''Calculate accuracy (% passed).'''
        if not results:
            return 0.0
        passed = sum(1 for r in results if r['passed'])
        return passed / len(results)

    @staticmethod
    def check_hallucination(output: str, context: str) -> bool:
        '''Check if output contains unsupported claims.

        Simple version: check if output mentions facts not in context.
        '''
        # This is simplified - real version would be more sophisticated
        output_words = set(output.lower().split())
        context_words = set(context.lower().split())

        # If output has many words not in context, possible hallucination
        unique_words = output_words - context_words
        hallucination_ratio = len(unique_words) / len(output_words) if output_words else 0

        return hallucination_ratio > 0.7  # Threshold

    @staticmethod
    def relevance_score(query: str, response: str) -> float:
        '''Calculate relevance (0-1).'''
        query_words = set(query.lower().split())
        response_words = set(response.lower().split())

        # Jaccard similarity
        intersection = query_words & response_words
        union = query_words | response_words

        return len(intersection) / len(union) if union else 0.0

metrics = EvaluationMetrics()

# Example
query = "What is machine learning?"
response = "Machine learning is a type of AI that learns from data"
print(f'Relevance: {metrics.relevance_score(query, response):.2f}')

Relevance: 0.15


## Part 3: Test Suite

Run all tests and report results.

In [ ]:
class AITestSuite:
    '''Test suite for AI systems.'''

    def __init__(self, name: str):
        self.name = name
        self.test_cases = []
        self.results = []

    def add_test(self, test_case: TestCase):
        '''Add a test case.'''
        self.test_cases.append(test_case)

    def run_tests(self, ai_function: callable) -> Dict[str, Any]:
        '''Run all tests.

        Args:
            ai_function: Function that takes input and returns output
        '''
        self.results = []

        for test in self.test_cases:
            # Get AI output
            actual = ai_function(test.input)

            # Evaluate
            result = test.evaluate(actual)
            self.results.append(result)

        # Calculate metrics
        accuracy = EvaluationMetrics.accuracy(self.results)

        return {
            'total_tests': len(self.test_cases),
            'passed': sum(1 for r in self.results if r['passed']),
            'failed': sum(1 for r in self.results if not r['passed']),
            'accuracy': accuracy
        }

    def report(self):
        '''Print test report.'''
        print(f'\n{"="*60}')
        print(f'Test Suite: {self.name}')
        print(f'{"="*60}')

        for i, result in enumerate(self.results, 1):
            status = '✓ PASS' if result['passed'] else '✗ FAIL'
            print(f"\nTest {i}: {status}")
            print(f"  Input: {result['input']}")
            print(f"  Expected: {result['expected']}")
            print(f"  Actual: {result['actual'][:50]}...")

        # Summary
        accuracy = EvaluationMetrics.accuracy(self.results)
        print(f"\n{'-'*60}")
        print(f"Accuracy: {accuracy*100:.1f}%")
        print(f"Passed: {sum(1 for r in self.results if r['passed'])}/{len(self.results)}")

# Example usage
suite = AITestSuite('Digital Twin Tests')

for tc in test_cases:
    suite.add_test(tc)

print(f'\nTest suite ready with {len(test_cases)} tests')


Test suite ready with 3 tests


## Part 4: Testing Your Digital Twin

Apply tests to your AI.

In [ ]:
# Mock AI for demo
def mock_ai(query: str) -> str:
    '''Simple mock AI for testing.'''
    if '2 + 2' in query:
        return 'The answer is 4'
    elif 'created' in query:
        return 'I am a digital twin assistant'
    elif 'weather' in query:
        return "I don't have access to current weather data"
    return "I don't understand."

# Run tests
results = suite.run_tests(mock_ai)
suite.report()

print(f"\n\nOverall Accuracy: {results['accuracy']*100:.1f}%")


Test Suite: Digital Twin Tests

Test 1: ✓ PASS
  Input: What is 2 + 2?
  Expected: 4
  Actual: The answer is 4...

Test 2: ✓ PASS
  Input: Who created you?
  Expected: I am a digital twin
  Actual: I am a digital twin assistant...

Test 3: ✓ PASS
  Input: What's the weather?
  Expected: I don't have access to current weather
  Actual: I don't have access to current weather data...

------------------------------------------------------------
Accuracy: 100.0%
Passed: 3/3


Overall Accuracy: 100.0%


## Exercise 1: Create Your Test Suite

**Task**: Create 10 test cases for YOUR digital twin.

1. Cover different capabilities
2. Include edge cases
3. Test for hallucinations
4. Run the suite
5. Improve your twin based on failures

In [ ]:
# YOUR CODE HERE

my_suite = AITestSuite('My Digital Twin Tests')

# Add your test cases
# my_suite.add_test(TestCase("input", "expected", "criteria"))

# Run and report
# results = my_suite.run_tests(your_ai_function)
# my_suite.report()

## Summary

### What We Learned:
- Test cases = input + expected + criteria
- Metrics: accuracy, hallucination, relevance
- Test suites automate quality checks
- Regular testing catches regressions

### Next: Week 3.3 - Guardrails & Safety